# ETRS923 – TP3 : Classification météo (MeteoStat) → TensorFlow → ONNX  (v2)
Partie 1 du TP (Colab) + export du modèle pour MATLAB/ThingSpeak (partie 2) et STM32N6 / X-CUBE-AI (partie 3).

## Entrées du réseau (12 au maximum)
| # | Entrée | Source sur le Cloud / la carte | Remarque |
|---|--------|-------------------------------|----------|
| 1 | `temp` (°C) | capteur | |
| 2 | `rhum` (%) | capteur | |
| 3 | `pres` (hPa, **niveau de la mer**) | capteur de pression + conversion | MeteoStat donne la pression ramenée au niveau de la mer ; la carte mesure la pression *à la station* (≈955 hPa à Annecy) → il faut convertir (cellule 7) |
| 4 | `dpres3h` (hPa) | historique ThingSpeak / buffer sur la carte | pression(t) − pression(t−3 h), très utile pour la pluie. Optionnelle (`USE_TENDANCE`) |
| 5 | `spread` (°C) | calculé | température − point de rosée (formule de Magnus) |
| 6-8 | `lat`, `lon`, `alt` | constantes de la carte | GPS + altitude |
| 9-10 | `doy_sin`, `doy_cos` | date | jour de l'année en sin/cos (cyclique) |
| 11-12 | `hour_sin`, `hour_cos` | heure **UTC** | heure en sin/cos (cyclique : 23 h est proche de 0 h) |

Sorties : une probabilité par classe (3, 5 ou 9 selon le schéma). La normalisation est intégrée au modèle.

In [ ]:
!pip -q install "meteostat<2" tf2onnx onnx onnxruntime

In [ ]:
import numpy as np, pandas as pd, json, time
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print(tf.__version__)

## 1. Données MeteoStat
Le champ `coco` (code météo 1 à 27) sert d'étiquette. On télécharge 10 villes, on régularise la série à 1 h (pour calculer correctement la tendance sur 3 h) et on garde les lignes complètes.

In [ ]:
from meteostat import Point, Hourly
from datetime import datetime

LOCS = {
    "Annecy": (45.90, 6.12, 448), "Chambery": (45.57, 5.92, 270),
    "Lyon": (45.76, 4.84, 200), "Grenoble": (45.19, 5.73, 214),
    "Paris": (48.85, 2.35, 35), "Marseille": (43.30, 5.37, 12),
    "Brest": (48.39, -4.49, 30), "Lille": (50.63, 3.07, 25),
    "Strasbourg": (48.58, 7.75, 150), "Toulouse": (43.60, 1.44, 150),
}
start, end = datetime(2019, 1, 1), datetime(2024, 12, 31, 23)

frames = []
for name, (lat, lon, alt) in LOCS.items():
    d = Hourly(Point(lat, lon, alt), start, end).fetch()
    d = d.asfreq("h")                      # une ligne par heure, trous = NaN
    d["dpres3h"] = d["pres"] - d["pres"].shift(3)
    d["city"] = name; d["lat"] = lat; d["lon"] = lon; d["alt"] = alt
    frames.append(d)
    print(f"{name:12s} {len(d):6d} lignes, coco dispo: {d['coco'].notna().mean():.0%}")
raw = pd.concat(frames)
raw.head()

## 2. Construction des entrées
Toutes les formules ci-dessous devront être **reproduites à l'identique** dans MATLAB et en C.

In [ ]:
def dewpoint(t, rh):
    # Magnus : point de rosée en °C
    a, b = 17.62, 243.12
    g = np.log(np.clip(rh, 1, 100) / 100) + a * t / (b + t)
    return b * g / (a - g)

def to_sealevel(p_station, t, alt):
    # pression station (hPa) -> pression ramenée au niveau de la mer (hPa)
    return p_station * (1 - 0.0065 * alt / (t + 0.0065 * alt + 273.15)) ** -5.257

d = raw.dropna(subset=["temp", "rhum", "pres", "coco"]).copy()
idx = d.index
doy = np.asarray(idx.dayofyear); hour = np.asarray(idx.hour)
d["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
d["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
d["hour_sin"] = np.sin(2 * np.pi * hour / 24)
d["hour_cos"] = np.cos(2 * np.pi * hour / 24)
d["spread"] = d["temp"] - dewpoint(d["temp"], d["rhum"])
d["coco"] = d["coco"].astype(int)
df = d

FEATURES_ALL = ["temp", "rhum", "pres", "dpres3h", "spread", "lat", "lon", "alt",
                "doy_sin", "doy_cos", "hour_sin", "hour_cos"]
USE_TENDANCE = True      # False si la carte ne peut pas garder 3 h d'historique
FEATURES = [f for f in FEATURES_ALL if USE_TENDANCE or f != "dpres3h"]
print(len(df), "lignes ; entrées :", len(FEATURES), FEATURES)
df["coco"].value_counts().sort_index()

## 3. Schémas de classes (3 / 5 / 9 classes)
Test = année 2024, entraînement = 2019-2023 (**découpage temporel** : en découpage aléatoire, des heures voisines très corrélées se retrouvent des deux côtés et gonflent l'accuracy).

In [ ]:
SCHEMES = {
 "3 classes": {
    "Clair": [1, 2], "Nuageux": [3, 4],
    "Precipitations": [7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22]},
 "5 classes": {
    "Clair": [1, 2], "Nuageux": [3, 4], "Pluie": [7,8,9,10,11,17,18],
    "Neige": [12,13,14,15,16,19,20,21,22], "Brouillard": [5, 6]},
 "9 classes": {
    "Clair": [1], "Peu nuageux": [2], "Part. nuageux": [3], "Couvert": [4],
    "Brouillard": [5, 6], "Pluie": [7,8,9,10,11], "Averses": [17, 18],
    "Neige": [12,13,14,15,16,19,20,21,22], "Orage": [23, 25, 26, 27]},
}

def make_dataset(scheme, features, min_count=200):
    code2name = {c: n for n, cs in SCHEMES[scheme].items() for c in cs}
    dd = df[df["coco"].isin(code2name)].dropna(subset=list(features)).copy()
    dd["label"] = dd["coco"].map(code2name)
    vc = dd["label"].value_counts()
    dd = dd[dd["label"].isin(vc[vc >= min_count].index)]
    names = sorted(dd["label"].unique())
    y = dd["label"].map({n: i for i, n in enumerate(names)}).values
    test = np.asarray(dd.index.year == 2024)
    return dd[list(features)].values.astype("float32"), y, names, test

for s in SCHEMES:
    X, y, names, test = make_dataset(s, FEATURES)
    print(s, "->", len(X), "échantillons (train", (~test).sum(), "/ test", test.sum(), ")", names)

## 4. Entraînement et étude du compromis classes / taille / accuracy

In [ ]:
def build_model(n_in, n_out, units, Xtrain):
    norm = keras.layers.Normalization()
    norm.adapt(Xtrain)
    layers = [keras.layers.Input((n_in,)), norm]
    for u in units:
        layers.append(keras.layers.Dense(u, activation="relu"))
    layers.append(keras.layers.Dense(n_out, activation="softmax"))
    return keras.Sequential(layers)

def run(scheme, units, features=None, lr=1e-3, batch=1024, epochs=15, verbose=0):
    t0 = time.perf_counter()
    features = features or FEATURES
    X, y, names, test = make_dataset(scheme, features)
    Xtr, ytr, Xte, yte = X[~test], y[~test], X[test], y[test]
    p = np.random.RandomState(SEED).permutation(len(Xtr))   # mélange avant validation_split
    Xtr, ytr = Xtr[p], ytr[p]
    classes = np.unique(ytr)
    cw = dict(zip(classes, compute_class_weight("balanced", classes=classes, y=ytr)))
    m = build_model(X.shape[1], len(names), units, Xtr)
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    es = keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_loss")
    m.fit(Xtr, ytr, validation_split=0.15, epochs=epochs, batch_size=batch,
          class_weight=cw, callbacks=[es], verbose=verbose)
    pred = m.predict(Xte, verbose=0).argmax(1)
    acc = (pred == yte).mean()
    f1 = f1_score(yte, pred, average="macro")
    info = dict(schema=scheme, archi=str(units), entrees=X.shape[1], sorties=len(names),
                params=m.count_params(), taille_ko_float32=round(m.count_params()*4/1024, 1),
                accuracy=round(acc, 4), f1_macro=round(f1, 4),
                duree_s=round(time.perf_counter() - t0))
    return m, info, (Xte, yte, names)

ARCHS = [(16,), (32, 16), (64, 32)]
results = []
for s in SCHEMES:
    for a in ARCHS:
        _, r, _ = run(s, a)
        results.append(r); print(r)
res = pd.DataFrame(results)
res

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for s, g in res.groupby("schema"):
    ax.plot(g["taille_ko_float32"], g["accuracy"], "o-", label=s)
ax.set_xlabel("Taille du modèle (Ko, float32)"); ax.set_ylabel("Accuracy (test 2024)")
ax.set_title("Compromis classes / taille / accuracy"); ax.legend(); ax.grid(alpha=.3)
plt.show()

### Quelles entrées valent le coup ? (ablation)
On ajoute les groupes d'entrées un par un (schéma 5 classes, réseau (32, 16)). Cela justifie chaque capteur ou calcul demandé à la carte.

In [ ]:
ABL = {
 "A: temp, rhum, pres":            ["temp", "rhum", "pres"],
 "B: + date/heure (sin/cos)":      ["temp", "rhum", "pres", "doy_sin", "doy_cos", "hour_sin", "hour_cos"],
 "C: + GPS + altitude":            ["temp", "rhum", "pres", "doy_sin", "doy_cos", "hour_sin", "hour_cos", "lat", "lon", "alt"],
 "D: + écart au point de rosée":   ["temp", "rhum", "pres", "doy_sin", "doy_cos", "hour_sin", "hour_cos", "lat", "lon", "alt", "spread"],
 "E: + tendance de pression 3 h":  FEATURES_ALL,
}
rows = []
for k, feats in ABL.items():
    _, r, _ = run("5 classes", (32, 16), features=feats)
    r["jeu d'entrées"] = k; rows.append(r)
abl = pd.DataFrame(rows)[["jeu d'entrées", "entrees", "params", "accuracy", "f1_macro"]]
abl

### Hyperparamètres (question 1.b)
On fait varier le learning rate et la taille de batch (schéma 5 classes) puis on **justifie** le choix final dans le compte rendu.

In [ ]:
rows = []
for lr in [1e-2, 1e-3, 1e-4]:
    for bs in [256, 1024]:
        _, r, _ = run("5 classes", (32, 16), lr=lr, batch=bs)
        r.update(lr=lr, batch=bs); rows.append(r)
pd.DataFrame(rows)[["lr", "batch", "archi", "entrees", "params", "accuracy", "f1_macro"]]

## 5. Modèle final
Modifier `CHOIX_SCHEMA` / `CHOIX_ARCHI` selon vos résultats.

In [ ]:
CHOIX_SCHEMA = "5 classes"
CHOIX_ARCHI = (32, 16)

model, r, (Xte, yte, names) = run(CHOIX_SCHEMA, CHOIX_ARCHI, verbose=1)
print(r)
model.summary()

pred = model.predict(Xte, verbose=0).argmax(1)
ConfusionMatrixDisplay(confusion_matrix(yte, pred, labels=range(len(names))),
                       display_labels=names).plot(xticks_rotation=45)
plt.show()

## 6. Export TensorFlow → ONNX
Entrée nommée `input`, forme `(None, N)` en float32, avec N = nombre d'entrées, dans l'ordre de `FEATURES` (sauvegardé dans `features.json`).

In [ ]:
import tf2onnx, onnx, onnxruntime as ort

N_FEAT = len(FEATURES)

# On fond la normalisation dans la 1re couche Dense : (x*scale + shift) puis Dense
# == Dense aux poids modifiés. Le réseau exporté n'a plus que Dense/ReLU/Softmax
# (pas de Sub/Sqrt/Div) : plus simple pour ONNX, MATLAB et X-CUBE-AI.
norm = model.layers[0]
z = np.zeros((1, N_FEAT), "float32"); o = np.ones((1, N_FEAT), "float32")
shift = np.array(norm(z))[0]
scale = np.array(norm(o))[0] - shift
dense = [l for l in model.layers if isinstance(l, keras.layers.Dense)]
export = keras.Sequential([keras.layers.Input((N_FEAT,))] +
                          [keras.layers.Dense(l.units, activation=l.activation) for l in dense])
for i, (le, l) in enumerate(zip(export.layers, dense)):
    W, b = l.get_weights()
    if i == 0:
        b = b + shift @ W
        W = W * scale[:, None]
    le.set_weights([W, b])
print("Écart max modèle / modèle sans normalisation :",
      np.abs(export.predict(Xte[:200], verbose=0) - model.predict(Xte[:200], verbose=0)).max())

spec = (tf.TensorSpec((None, N_FEAT), tf.float32, name="input"),)

@tf.function(input_signature=spec)
def serve(x):
    return export(x)

model.save("meteo_model.keras")
onnx_model, _ = tf2onnx.convert.from_function(serve, input_signature=spec, opset=13, output_path="meteo.onnx")
print("Entrées :", [i.name for i in onnx_model.graph.input], "Sorties :", [o.name for o in onnx_model.graph.output])
assert len(onnx_model.graph.input) == 1, "L'ONNX a des entrées en trop"

sess = ort.InferenceSession("meteo.onnx")
iname = sess.get_inputs()[0].name
out_onnx = sess.run(None, {iname: Xte[:200]})[0]
out_keras = model.predict(Xte[:200], verbose=0)
print("Écart max Keras/ONNX :", np.abs(out_onnx - out_keras).max())

t = time.perf_counter()
for _ in range(1000): sess.run(None, {iname: Xte[:1]})
print("Inférence CPU Colab : %.3f ms / échantillon" % (time.perf_counter() - t))

## 7. Fonction de référence à reproduire sur MATLAB / STM32
Cette fonction montre exactement comment passer des **mesures brutes** aux entrées du réseau. La carte mesure la pression *à la station* : elle doit d'abord la convertir au niveau de la mer.

In [ ]:
def make_features(temp, rhum, p_station, dpres3h, lat, lon, alt, utc):
    # utc : datetime en UTC (pas l'heure locale !)
    pres = to_sealevel(p_station, temp, alt)
    doy = utc.timetuple().tm_yday
    v = {"temp": temp, "rhum": rhum, "pres": pres, "dpres3h": dpres3h,
         "spread": temp - dewpoint(temp, rhum), "lat": lat, "lon": lon, "alt": alt,
         "doy_sin": np.sin(2*np.pi*doy/365.25), "doy_cos": np.cos(2*np.pi*doy/365.25),
         "hour_sin": np.sin(2*np.pi*utc.hour/24), "hour_cos": np.cos(2*np.pi*utc.hour/24)}
    return np.array([v[f] for f in FEATURES], dtype="float32")

# Vérification : la fonction retrouve la ligne du jeu de données (en repartant de la pression station)
r0 = df.dropna(subset=FEATURES).iloc[1000]
p_stat = r0["pres"] / (1 - 0.0065 * r0["alt"] / (r0["temp"] + 0.0065 * r0["alt"] + 273.15)) ** -5.257
f = make_features(r0["temp"], r0["rhum"], p_stat, r0["dpres3h"], r0["lat"], r0["lon"], r0["alt"], r0.name.to_pydatetime())
print("Écart max :", np.abs(f - r0[FEATURES].values.astype("float32")).max())
print("Exemple Annecy : 955 hPa à 15 °C ->", round(float(to_sealevel(955, 15, 448)), 1), "hPa niveau mer")

In [ ]:
with open("labels.json", "w") as fo: json.dump(names, fo, ensure_ascii=False)
with open("features.json", "w") as fo:
    json.dump({"features": FEATURES, "classes": names,
               "annecy": {"lat": 45.90, "lon": 6.12, "alt": 448},
               "notes": "heure en UTC ; pres = niveau de la mer ; dpres3h = pres(t) - pres(t-3h)"},
              fo, ensure_ascii=False, indent=1)
pd.DataFrame(Xte[:50], columns=FEATURES).assign(label_vrai=[names[i] for i in yte[:50]]).to_csv("echantillons_test.csv", index=False)

from google.colab import files
for fn in ["meteo.onnx", "meteo_model.keras", "labels.json", "features.json", "echantillons_test.csv"]:
    files.download(fn)

## 8. Suite du TP
- **MATLAB / ThingSpeak** : `net = importNetworkFromONNX("meteo.onnx", InputDataFormats="BC");` puis `predict(net, dlarray(single(x), "BC"))` avec `x` = vecteur de `features.json` **dans le même ordre**. (`importONNXNetwork` échoue : format d'entrée inconnu.) Utiliser l'heure UTC et convertir la pression en niveau de la mer.
- **STM32N6** : charger `meteo.onnx` dans X-CUBE-AI, reproduire `make_features` en C (`logf`, `sinf`, `cosf`, `powf`) et garder un buffer pour la pression à t−3 h.
- Comparer ensuite accuracy et temps d'exécution Cloud vs Edge (3.c) avec les chiffres de ce notebook.